# ECI data pipeline

Single-notebook, refreshable data prep for the ECI Bayesian model. Run all
cells top-to-bottom to:

1. **Fetch** Epoch AI's bulk benchmark ZIP (CC-BY) into a dated snapshot folder.
2. **Inventory** what's inside and diff against the previous snapshot.
3. **Load + normalize** scores into a long-format table.
4. **Canonicalize** benchmark and model names.
5. **Dedup** exact duplicates (and resolve disagreements by `max`).
6. **Attach metadata** (category) from a human-maintained file.
7. **Filter** (optional curated exclusions; off by default).
8. **Humans** — apply the canonicalization map to `1_data/curated/human_baselines.csv`.
9. **Assemble** the final `benchmarks_merged.csv` and a `pipeline_report.md` provenance summary.

Outputs land in `1_data/1_pipeline/output/`; section 10 copies
`benchmarks_merged.csv` and `human_baselines.csv` into `1_data/processed/` and
`1_data/curated/`. Read `output/pipeline_report.md`, then rebuild the lineage map
(`1_data/3_build_lineage_map.py`) and the SOTA list
(`1_data/2_compute_sota.py`) before re-fitting. Revert with
`git checkout 1_data/processed 1_data/curated`.


## 00 — Setup

Imports, paths, parameters.

In [1]:
import json
import hashlib
import shutil
import urllib.request
import re
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

# Notebook is expected to run from 1_data/1_pipeline/
PIPELINE_DIR = Path.cwd()
if PIPELINE_DIR.name != "pipeline":
    print(f"WARN: cwd is {PIPELINE_DIR}, expected to end in /pipeline. Paths may be wrong.")

PROJECT_ROOT    = PIPELINE_DIR.parents[1]
CANONICAL_DIR   = PIPELINE_DIR / "canonical"
SNAPSHOTS_DIR   = PIPELINE_DIR / "snapshots"
INTERMEDIATE_DIR = PIPELINE_DIR / "intermediate"
OUTPUT_DIR      = PIPELINE_DIR / "output"
CURATED_DIR     = PROJECT_ROOT / "data" / "curated"
PROCESSED_DIR   = PROJECT_ROOT / "data" / "processed"

for p in [SNAPSHOTS_DIR, INTERMEDIATE_DIR, OUTPUT_DIR, CANONICAL_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# --- Parameters ---
EPOCH_ZIP_URL = "https://epoch.ai/data/benchmark_data.zip"

# The published ZIP lags the site on benchmark VERSION changes: six weeks after
# the 2026-06-12 FrontierMath v2 release it still exported only pre-update runs,
# and its per-benchmark CSVs carry no version column to tell the series apart.
# This is the feed the epoch.ai benchmark pages actually read. It names each run
# in a `task` column and stamps `task version`, so a specific problem-set
# version can be selected. Section 03d reads ALL Epoch internal evals from here,
# so they share one transport; the ZIP supplies the external benchmarks.
EPOCH_LIVE_URL = "https://epoch.ai/data/benchmarks.csv"

# Curated "easy-for-humans" exclusions are a MODELING concern, applied at fit time
# by data.py (load_excluded_benchmarks), NOT during data generation. The pipeline
# emits every benchmark it can assemble; keep this False so the processed file is
# complete and the exclusion policy lives in one place (data.py).
APPLY_CURATED_EXCLUSIONS = False

# Fallback policy for the (rare) case of disagreeing duplicates on (model, benchmark).
# Exact-row duplicates are always dropped regardless.
DEDUP_POLICY = "max"

# Match data.py's open-interval clip used for the Beta likelihood.
ECI_EPS = 1e-3

print(f"PIPELINE_DIR = {PIPELINE_DIR}")
print(f"PROJECT_ROOT = {PROJECT_ROOT}")


PIPELINE_DIR = /Users/yassineessifi/Desktop/ECI_Bayesian/data/pipeline
PROJECT_ROOT = /Users/yassineessifi/Desktop/ECI_Bayesian


## 01 — Fetch

Download Epoch's bulk ZIP into `snapshots/YYYY-MM-DD/`. Idempotent within a
day (won't re-download if today's snapshot exists). Writes `provenance.json`
with the URL, sha256, fetch time, license, and attribution.

The three Epoch-missing sources (PRBench Finance, OSUniverse, Lech Mazur)
are not auto-fetched in v1 — drop them into the snapshot folder manually if
you want them included.


In [2]:
today = datetime.now(timezone.utc).strftime("%Y-%m-%d")
snapshot_dir = SNAPSHOTS_DIR / today
epoch_dir    = snapshot_dir / "epoch"
zip_path     = snapshot_dir / "epoch_benchmark_data.zip"
prov_path    = snapshot_dir / "provenance.json"

if snapshot_dir.exists() and prov_path.exists() and any(epoch_dir.rglob("*.csv")):
    print(f"Snapshot already exists at {snapshot_dir} — re-using.")
else:
    snapshot_dir.mkdir(parents=True, exist_ok=True)
    print(f"Downloading {EPOCH_ZIP_URL}")
    urllib.request.urlretrieve(EPOCH_ZIP_URL, zip_path)

    sha = hashlib.sha256(zip_path.read_bytes()).hexdigest()
    size = zip_path.stat().st_size
    print(f"  sha256: {sha}")
    print(f"  size:   {size:,} bytes")

    if epoch_dir.exists():
        shutil.rmtree(epoch_dir)
    epoch_dir.mkdir(parents=True)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(epoch_dir)
    n_files = sum(1 for _ in epoch_dir.rglob("*"))
    print(f"  unzipped {n_files} files to {epoch_dir}")

    prov = {
        "url":         EPOCH_ZIP_URL,
        "fetched_at":  datetime.now(timezone.utc).isoformat(),
        "sha256":      sha,
        "byte_size":   size,
        "license":     "CC-BY",
        "attribution": "Epoch AI",
    }
    prov_path.write_text(json.dumps(prov, indent=2))
    print(f"  wrote {prov_path.name}")

# --- Live benchmarks feed (version-tagged; used for FrontierMath v2) ---
# Fetched separately from the ZIP so a re-used snapshot folder still picks it up.
live_csv_path = snapshot_dir / "epoch_live_benchmarks.csv"
if live_csv_path.exists():
    print(f"Live feed already snapshotted at {live_csv_path.name} — re-using.")
else:
    print(f"Downloading {EPOCH_LIVE_URL}")
    urllib.request.urlretrieve(EPOCH_LIVE_URL, live_csv_path)
    live_sha = hashlib.sha256(live_csv_path.read_bytes()).hexdigest()
    print(f"  sha256: {live_sha}")
    print(f"  size:   {live_csv_path.stat().st_size:,} bytes")
    live_prov_path = snapshot_dir / "live_provenance.json"
    live_prov_path.write_text(json.dumps({
        "url":         EPOCH_LIVE_URL,
        "fetched_at":  datetime.now(timezone.utc).isoformat(),
        "sha256":      live_sha,
        "byte_size":   live_csv_path.stat().st_size,
        "license":     "CC-BY",
        "attribution": "Epoch AI",
        "used_for":    "all Epoch internal evals (section 03d)",
    }, indent=2))
    print(f"  wrote {live_prov_path.name}")

LATEST_LIVE_CSV = live_csv_path

LATEST_SNAPSHOT  = snapshot_dir
LATEST_EPOCH_DIR = epoch_dir

print("\nOptional supplementary sources — drop into the snapshot folder manually if available:")
print(f"  {snapshot_dir / 'prbench_finance.csv'}   (Scale SEAL — HTML scrape)")
print(f"  {snapshot_dir / 'osuniverse.csv'}         (agentsea/osuniverse — GitHub)")
print(f"  {snapshot_dir / 'lech_mazur_writing.csv'} (lechmazur/writing — GitHub)")


  sha256: 591b9f32b820c19e8e7684cb193a1b6efa05378dd09ad349e7c82f240bd5f07b
  size:   450,497 bytes
  unzipped 77 files to /Users/yassineessifi/Desktop/ECI_Bayesian/1_data/1_pipeline/snapshots/2026-08-07/epoch
  wrote provenance.json


  sha256: c703eacd3664655516edce89180810065b731a114a5b373c78d45010260f0512
  size:   2,778,871 bytes
  wrote live_provenance.json

Optional supplementary sources — drop into the snapshot folder manually if available:
  /Users/yassineessifi/Desktop/ECI_Bayesian/1_data/1_pipeline/snapshots/2026-08-07/prbench_finance.csv   (Scale SEAL — HTML scrape)
  /Users/yassineessifi/Desktop/ECI_Bayesian/1_data/1_pipeline/snapshots/2026-08-07/osuniverse.csv         (agentsea/osuniverse — GitHub)
  /Users/yassineessifi/Desktop/ECI_Bayesian/1_data/1_pipeline/snapshots/2026-08-07/lech_mazur_writing.csv (lechmazur/writing — GitHub)


## 02 — Inventory

List the CSVs in the latest snapshot with their shape and column names so
you can verify what Epoch shipped this week, and adjust the column-alias
map in section 03 if Epoch renamed something.

Also diffs against the previous snapshot folder if one exists.


In [3]:
print(f"Inventorying {LATEST_EPOCH_DIR}\n")

csv_files = sorted(LATEST_EPOCH_DIR.rglob("*.csv"))
print(f"Found {len(csv_files)} CSV file(s):\n")

for f in csv_files:
    try:
        df = pd.read_csv(f, low_memory=False, nrows=5000)
        print(f"{f.relative_to(LATEST_EPOCH_DIR)}: {len(df):,}+ rows × {len(df.columns)} cols")
        print(f"  cols: {list(df.columns)}")
    except Exception as e:
        print(f"{f.name}: read failed ({e})")
    print()

# Diff vs previous snapshot
prev_snapshots = sorted(
    [p for p in SNAPSHOTS_DIR.iterdir() if p.is_dir() and p != LATEST_SNAPSHOT]
)
if prev_snapshots:
    prev = prev_snapshots[-1]
    print(f"Diff vs previous snapshot ({prev.name}):")
    prev_files = {f.name for f in (prev / "epoch").rglob("*.csv")}
    curr_files = {f.name for f in LATEST_EPOCH_DIR.rglob("*.csv")}
    new  = curr_files - prev_files
    gone = prev_files - curr_files
    if new:  print(f"  new files: {sorted(new)}")
    if gone: print(f"  removed:   {sorted(gone)}")
    if not new and not gone:
        print("  (no file-level changes)")
else:
    print("(no previous snapshot to diff against)")


Inventorying /Users/yassineessifi/Desktop/ECI_Bayesian/1_data/1_pipeline/snapshots/2026-08-07/epoch

Found 75 CSV file(s):

additional_eci_data/eci_benchmark_difficulties_and_slopes.csv: 57+ rows × 5 cols
  cols: ['benchmark_name', 'is_anchor', 'benchmark_release_date', 'edi', 'estimated_slope_scaled']

adversarial_nli_external.csv: 19+ rows × 16 cols
  cols: ['Model version', 'Score', 'Release date', 'Organization', 'Country', 'Training compute (FLOP)', 'Training compute notes', 'Name', 'R1', 'R2', 'R3', 'Shots', 'Source', 'Source link', 'Notes', 'id']

aider_polyglot_external.csv: 77+ rows × 16 cols
  cols: ['Model version', 'Percent correct', 'Release date', 'Organization', 'Country', 'Training compute (FLOP)', 'Training compute notes', 'Percent using correct edit format', 'Edit format', 'Cost', 'Source', 'Source link', 'Notes', 'Time per case (seconds)', 'Date of evaluation', 'id']

ale_bench_external.csv: 106+ rows × 15 cols
  cols: ['Model version', 'Performance', 'Release date',

## 03 — Load + normalize

Epoch ships one CSV per benchmark with the benchmark identity in the
filename and the score column varying per file (`EM`, `Accuracy`,
`mean_score`, `Mean score`, `Score (AVG@5)`, `Pass@1 score`, etc.). The
loader uses an explicit per-file spec map (`FILE_SPEC`):

```
filename → (canonical_benchmark_name, score_column, divisor)
```

`divisor` rescales the column onto `[0, 1]`. Most Epoch columns are
already on that scale even when named like a percentage (`% Score`,
`Overall pass (%)`) — `divisor=1.0` for those. A handful are genuine
percentages or out-of-10 (`Percent correct`, `Global average`, `Mean
score`) and get a real divisor.

Files not in `FILE_SPEC` are listed in `FILE_SKIP` with a reason, or
flagged as unknown. Edit the spec when Epoch adds a new file.


In [4]:
# Per-file spec — adjust when Epoch publishes new files.
# Format: filename: (benchmark_name, score_column, divisor)
FILE_SPEC = {
    # --- External (model-developer reported) ---
    "adversarial_nli_external.csv":    ("Adversarial NLI",        "Score",                  1.0),
    "aider_polyglot_external.csv":     ("Aider Polyglot",         "Percent correct",        100.0),
    "apex_agents_external.csv":        ("APEX Agents",            "Pass@1 score",           1.0),
    "arc_agi_external.csv":            ("ARC-AGI",                "Score",                  1.0),
    "arc_agi_2_external.csv":          ("ARC-AGI-2",              "Score",                  1.0),
    "arc_ai2_external.csv":            ("ARC (AI2)",              "Challenge score",        1.0),
    "balrog_external.csv":             ("BALROG",                 "Average progress",       1.0),
    "bbh_external.csv":                ("BIG-Bench Hard (BBH)",   "Average",                1.0),
    "bool_q_external.csv":             ("BoolQ",                  "Score",                  1.0),
    "cad_eval_external.csv":           ("CAD-Eval",               "Overall pass (%)",       1.0),
    "common_sense_qa_2_external.csv":  ("CSQA2",                  "Score",                  1.0),
    "cybench_external.csv":            ("Cybench",                "Unguided % Solved",      1.0),
    "deepresearchbench_external.csv":  ("DeepResearchBench",      "Average score",          1.0),
    "fictionlivebench_external.csv":   ("Fiction.LiveBench",      "120k token score",       1.0),
    "gsm8k_external.csv":              ("GSM8K",                  "EM",                     1.0),
    "gso_external.csv":                ("GSO-Bench",              "Score OPT@1",            1.0),
    "hella_swag_external.csv":         ("HellaSwag",              "Overall accuracy",       1.0),
    "hle_external.csv":                ("Humanity\'s Last Exam", "Accuracy",               1.0),
    "lambada_external.csv":            ("LAMBADA",                "Score",                  1.0),
    "lech_mazur_writing_external.csv": ("Lech Mazur Writing",     "Mean score",             10.0),
    "live_bench_external.csv":         ("LiveBench",              "Global average",         100.0),
    "metr_time_horizons_external.csv": ("METR Time Horizons",     "average_score",          1.0),
    "mmlu_external.csv":               ("MMLU",                   "EM",                     1.0),
    "open_book_qa_external.csv":       ("OpenBookQA",             "Accuracy",               1.0),
    "os_world_external.csv":           ("OS World (Screenshot)",  "Score",                  100.0),
    "piqa_external.csv":               ("PIQA",                   "Score",                  1.0),
    "posttrainbench_external.csv":     ("PostTrainBench",         "Average (%)",            1.0),
    "science_qa_external.csv":         ("ScienceQA",              "Score",                  1.0),
    "simplebench_external.csv":        ("SimpleBench",            "Score (AVG@5)",          1.0),
    "superglue_external.csv":          ("SuperGLUE",              "Score",                  1.0),
    "terminalbench_external.csv":      ("TerminalBench",          "Accuracy mean",          1.0),
    "the_agent_company_external.csv":  ("The Agent Company",      "% Score",                1.0),
    "trivia_qa_external.csv":          ("TriviaQA",               "EM",                     1.0),
    "video_mme_external.csv":          ("Video-MME",              "Overall (no subtitles)", 1.0),
    "vpct_external.csv":               ("VPCT",                   "Correct",                1.0),
    "weirdml_external.csv":            ("WeirdML",                "Accuracy",               1.0),
    "wino_grande_external.csv":        ("WinoGrande",             "Accuracy",               1.0),

    # --- New Epoch files (added 2026-07-08) ---
    "scicode_external.csv":            ("SciCode",                "Score",                  1.0),
    "cursorbench_external.csv":        ("CursorBench",            "Score",                  1.0),
    # Epoch stopped publishing "Diamond score" and now reports "Main score"
    # (2026-07-27 ZIP). NOT a rename: on the 3 models present in both
    # snapshots, Main runs 6-8x higher (gpt-5.5 0.063 -> 0.430), so it is an
    # easier subset. Diamond is gone upstream, so it cannot be kept current.
    # Like FrontierMath v2, the benchmark name now refers to a different
    # instrument: FrontierCode difficulty is not comparable across this change.
    "frontiercode_external.csv":       ("FrontierCode",           "Main score",             1.0),
    "exploitbench_external.csv":       ("ExploitBench",           "Mean capability",        1.0),
    "critpt_external.csv":             ("CritPt",                 "Accuracy",               1.0),
    "rli_external.csv":                ("Remote Labor Index",     "Score",                  1.0),
    "cl_bench_external.csv":           ("CL-Bench",               "Overall",                1.0),
    "cl_bench_life_external.csv":      ("CL-Bench Life",          "Overall",                1.0),
    "forecastbench_external.csv":      ("ForecastBench",          "Overall score",          100.0),
    # EnigmaEval moved into the ZIP (2026-07-27), 76 rows with stderr and CI
    # bounds, same Scale leaderboard the SEAL scraper used to read. The board
    # stopped serving its table to the scraper (47 rows -> 5), so the ZIP is
    # now both the fuller and the reliable source. Dropped from SEAL_SPEC (03c).
    "enigma_eval_external.csv":        ("EnigmaEval",             "Accuracy",               1.0),
    # OS World 2: "Binary accuracy" = fraction of tasks fully solved (task
    # success rate) — the clean pass/fail [0,1] analog of the original OS World
    # "Score". "Partial score" (partial credit within a task) is deliberately
    # not used: noisier and less comparable across the index.
    "osworld_2_external.csv":          ("OS World 2",             "Binary accuracy",        1.0),

    # GeoBench: ingest the country-identification accuracy sub-metric ONLY.
    # The headline "ACW Avg Score" is a GeoGuessr points score (~0-5000, seen
    # 2131-4333) — NOT a [0,1] proportion, so it can't feed the Beta model.
    # "ACW Country %" is the one clean, full-coverage (n=32) [0,1] column —
    # the fraction of images whose country the model names correctly — and it
    # lines up with the 0.90 "top player" human baseline (best model 0.88).
    # Other splits (AVW/Rural/Urban/Photos) are sparse. See README "GeoBench".
    "geobench_external.csv":           ("GeoBench",               "ACW Country %",          1.0),

    # --- Un-skipped 2026-07-27: already [0,1], no transform needed ---
    # GDPval: blinded pairwise comparison against HUMAN EXPERT deliverables, a
    # fixed exogenous reference, NOT other models. The previous skip reason
    # ("win rate vs other models") was factually wrong. "Win Rate (%)" is the
    # strict-superiority measure (0.099-0.497); "Win + tie rate (%)" (0.123-0.709)
    # is the alternative if parity-with-expert should count as success.
    # NOTE both columns are already PROPORTIONS despite the "(%)" header: divisor 1.0.
    "gdpval_external.csv":             ("GDPval",                 "Win Rate (%)",           1.0),
    # ; it is a plain [0,1] score, 0.020-0.300.
    "gdp_pdf_external.csv":            ("GDP.pdf",                "GDP.pdf score",          1.0),
    # Weighted mean of three [0,1] sub-scores (Replay/Procedural/Audio), 0-0.745.
    # Caveat kept visible rather than filtered: glm-5.1 and MiniMax-M2.7 score 0
    # with Build failed=2/1 and Grading failed=143, i.e. harness failure rather
    # than measured incapability. They are ingested; treat those two rows with care.
    "gbaeval_external.csv":            ("GBAEval",                "Overall score",          1.0),
    # AlgoTune: harmonic-mean speedup (1.31-2.05x) against a FIXED reference
    # implementation (SciPy / sklearn / CVXPY), so no dependence on the model pool.
    # Rescaled by FILE_SCORE_TRANSFORM to the fraction of runtime eliminated.
    "algotune_external.csv":           ("AlgoTune",               "Score",                  1.0),

    # --- Previously unmapped external files (ingested 2026-07-27) ---
    # All present in the feed since at least the 2026-07-21 snapshot; they were
    # only ever flagged in cell output, never in pipeline_report.md (now fixed).
    # Every score column below is already a [0,1] proportion, so divisor 1.0.
    "proofbench_external.csv":         ("ProofBench",             "Accuracy",               1.0),
    "deepswe_external.csv":            ("DeepSWE",                "Pass@1",                 1.0),
    "surface_evolver_bench_external.csv": ("Surface Evolver Bench", "Mean score",           1.0),
    "blueprint_bench_2_external.csv":  ("BlueprintBench 2",       "Score",                  1.0),
    # SpatialViz-Bench and MindCube cover only 2024-01..2025-06 models (8 and 5
    # rows, no frontier releases), so they inform old-model abilities on a new
    # axis while adding no frontier coverage. Ingested on request; read their
    # loadings with the coverage/extrapolation caveat in mind.
    "spatialviz_bench_external.csv":   ("SpatialViz-Bench",       "Overall score",          1.0),
    "mindcube_external.csv":           ("MindCube",               "Overall score",          1.0),

    # --- Un-skipped 2026-07-29: dollar-denominated, rescaled below ---

}

# Files explicitly excluded with reason. Anything not in FILE_SPEC and not
# in FILE_SKIP gets flagged as "unrecognized".
FILE_SKIP = {
    # Routed through the live feed instead (section 03d), where every Epoch
    # internal eval is read, so the two transports cannot double-count it.
    "mystery_game_puzzles.csv": "internal eval; ingested from the live feed",
    "epoch_capabilities_index.csv":   "Epoch\'s own ECI output (downstream of this pipeline)",
    "eci_benchmark_difficulties_and_slopes.csv":
                                      "Epoch\'s own fitted IRT parameters (downstream of this pipeline)",
    # "Dominance" is the win rate against a random opponent drawn from the OTHER
    # MODELS on the leaderboard. It is in [0,1], so it passes a pure scale test,
    # but corr(Dominance, pool-normalized rank) = 0.977 over all 12 rows: it is a
    # reparametrized rank. Adding a 13th model changes every existing row, which
    # breaks the conditional independence the IRT likelihood assumes.
    "frontierswe_external.csv":       "pool-normalized rank in disguise (r=0.977 vs rank); breaks conditional independence",
    "webdev_arena_external.csv":      "Elo-style Arena Score, not [0,1]",
    # Retired 2026-08-04. 54 rows but only 3 informative takers, so the panel
    # cannot identify the row: the loading direction is set by whatever the rest
    # of the fit settles into, not by these scores. The dollar->proportion
    # transform also puts it on a scale no other benchmark shares.
    "vending_bench_2_external.csv":    "panel cannot identify the loading row (3 informative takers); non-commensurable dollar transform",
    "ale_bench_external.csv":         "Performance is a competition points/Elo score (~138-2041), not [0,1]",
    "README.md":                      "not a data file",
    # These six are byte-identical in the ZIP and in the live feed (same run ids,
    # same scores, identical model ids, 0 mismatches -- verified 2026-07-28). Routed
    # through the live feed so every Epoch INTERNAL eval has one transport, the same
    # one that carries FrontierMath's problem-set version and EBR-bench.
    "chess_puzzles.csv":              "Epoch internal eval; sourced from the live feed (03d)",
    "gpqa_diamond.csv":               "Epoch internal eval; sourced from the live feed (03d)",
    "math_level_5.csv":               "Epoch internal eval; sourced from the live feed (03d)",
    "otis_mock_aime_2024_2025.csv":   "Epoch internal eval; sourced from the live feed (03d)",
    "swe_bench_verified.csv":         "Epoch internal eval; sourced from the live feed (03d)",
    "simpleqa_verified.csv":          "Epoch internal eval; sourced from the live feed (03d)",
    # Pooled Brier/RPS loss. The 1-loss inversion is sound, but the FLOOR is not
    # derivable: a coin-flip binary forecaster scores Brier 0.25 (index 0.75), yet
    # the pooled score also weights a "normalized" numeric RPS 3x per question
    # (392 numeric vs 1515 binary) whose uninformative level is unpublished -- the
    # leaderboard carries no baseline row and the paper is "to follow". Parked
    # rather than shipping a guessed chance floor.
    "btf3_external.csv":              "pooled Brier/RPS loss; no published uninformative baseline for the floor",
    # The v1 problem set is DEFECTIVE, not merely differently scaled: the 2026-06-12
    # v2 release fixed errors in 42% of problems. A v1 score therefore mixes ability
    # with mis-keyed and unsolvable items, and that error is SYSTEMATIC (a correct
    # solution could be marked wrong), so a per-benchmark sigma_b cannot absorb it
    # and a separate D/A would not rescue it either. v2 re-ran only 41 of the 106
    # models v1 covered; the resulting coverage gap is accepted deliberately, on the
    # grounds that no measurement beats a wrong one. Section 03d supplies v2.
    "frontiermath.csv":               "v1 problem set defective (errors in 42% of problems, fixed in v2)",
    "frontiermath_tier_4.csv":        "v1 problem set defective (errors in 42% of problems, fixed in v2)",
}

# Score columns that are NOT already a [0,1] capability proportion, with the
# monotone rescaling that makes them one. Applied AFTER the divisor.
#
# A transform is only admissible if the reference it is measured against is
# EXOGENOUS (a fixed baseline, a published human figure, a library implementation).
# Anything measured against the other models on a leaderboard stays out entirely:
# no rescaling repairs it, because the quantity is not a property of the model.
# See FILE_SKIP for frontierswe/webdev_arena.
#
# AlgoTune: harmonic-mean speedup s >= 1 vs a fixed library implementation.
#   1 - 1/s is the fraction of the original runtime eliminated: s=1 (no improvement)
#   -> 0, s=2 -> 0.5, and the observed 1.31-2.05x -> 0.237-0.512, all interior.
#   Unbounded above in principle, so 1.0 means infinite speedup and is unreachable;
#   leave the 4PL ceiling inert at d=1 rather than inventing a finite cap.
FILE_SCORE_TRANSFORM = {
    "algotune_external.csv": (lambda x: 1.0 - 1.0 / x,  "1 - 1/speedup"),
}

# Standard meta-column rename map (Epoch is consistent across files).
META_RENAME = {
    "Model version":      "model_version",
    "Release date":       "release_date",
    "Organization":       "organization",
    "Source":             "source",
    "Source link":        "source_link",
    "Notes":              "notes",
    "id":                 "epoch_id",
}

# FILE_SKIP is tested first in the loop below, so an entry present in BOTH dicts
# is silently skipped and its FILE_SPEC line does nothing (this happened when
# AlgoTune was added). Fail loudly instead of shipping a dead spec line.
_both = sorted(set(FILE_SPEC) & set(FILE_SKIP))
if _both:
    raise ValueError(f"in both FILE_SPEC and FILE_SKIP (skip wins, spec is dead): {_both}")
_no_spec = sorted(set(FILE_SCORE_TRANSFORM) - set(FILE_SPEC))
if _no_spec:
    raise ValueError(f"FILE_SCORE_TRANSFORM entries with no FILE_SPEC entry: {_no_spec}")

frames = []
unrecognized = []
nan_dropped = []   # rows lost to a blank Model version, for the report

for f in sorted(LATEST_EPOCH_DIR.rglob("*.csv")):
    name = f.name
    if name in FILE_SKIP:
        print(f"  - SKIP {name}: {FILE_SKIP[name]}")
        continue
    if name not in FILE_SPEC:
        unrecognized.append(name)
        continue

    bench_name, score_col, divisor = FILE_SPEC[name]
    try:
        df = pd.read_csv(f, low_memory=False)
    except Exception as e:
        print(f"  ! {name}: read failed: {e}")
        continue

    if score_col not in df.columns:
        print(f"  ! {name}: expected score column {score_col!r} not found; has {list(df.columns)}")
        continue
    if "Model version" not in df.columns:
        print(f"  ! {name}: no 'Model version' column; has {list(df.columns)}")
        continue

    out = pd.DataFrame({
        "model_version": df["Model version"],
        "score":         pd.to_numeric(df[score_col], errors="coerce") / divisor,
        "benchmark":     bench_name,
    })

    if name in FILE_SCORE_TRANSFORM:
        fn, label = FILE_SCORE_TRANSFORM[name]
        out["score"] = fn(out["score"])
        print(f"    · {name}: rescaled {score_col!r} -> {label}")
    for upstream, canonical in META_RENAME.items():
        if upstream in df.columns and canonical not in {"model_version"}:
            out[canonical] = df[upstream]

    # --- Parse run-config labels out of Epoch's Name field (added 2026-07-05) ---
    # Epoch sometimes leaves `Model version` bare while the human-readable
    # `Name` carries the run configuration, e.g. "Gemini 3 Flash Preview
    # (Minimal)" with model_version "gemini-3-flash-preview". Left unparsed,
    # several real configs collapse onto one test-taker and the max-dedup then
    # mixes configs ACROSS benchmarks (found 2026-07-05: flash-3 ARC pair was
    # v1=Minimal vs v2=High and looked "impossible"). Parse the label into the
    # standard effort suffix whenever model_version doesn't already carry one.
    if "Name" in df.columns:
        # Effort label in the Name, e.g. "... (High)" / "... (x-high)".
        # An optional "Thinking," prefix inside the parenthetical is consumed:
        # "Opus 4.5 (Thinking, None)" states effort none exactly as "(none)"
        # does, and unparsed it left those rows on the BARE id (found
        # 2026-08-06 by 3_diagnostics/audit_model_names.py).
        cfg = (df["Name"].astype(str)
               .str.extract(r"(?i)\((?:thinking,\s*)?(minimal|low|medium|high|x-high|xhigh|max|none)\)\s*$",
                            expand=False)
               .str.lower().str.replace("x-high", "xhigh", regex=False))
        mv = out["model_version"].astype(str)
        # A model_version may ALREADY carry a config token. Only a bare id gets
        # the effort appended; an `_unknown` placeholder gets it substituted in;
        # a real effort suffix or a thinking-budget (`_32K`, `_120K`) is left
        # alone. Otherwise we stack suffixes into artefacts like `_32K_high` /
        # `_unknown_high` (found 2026-07-06 on APEX Agents).
        has_effort = mv.str.contains(r"_(?:minimal|low|medium|high|xhigh|max|none)$", regex=True)
        has_budget = mv.str.contains(r"_\d+K$", regex=True)
        placeholder = mv.str.contains(r"_unknown$", regex=True)
        bare = ~(has_effort | has_budget | placeholder)
        append  = cfg.notna() & bare
        replace = cfg.notna() & placeholder
        if int(append.sum()):
            out.loc[append, "model_version"] = mv[append] + "_" + cfg[append]
        if int(replace.sum()):
            out.loc[replace, "model_version"] = (
                mv[replace].str.replace(r"_unknown$", "", regex=True) + "_" + cfg[replace])
        n = int(append.sum()) + int(replace.sum())
        if n:
            print(f"    · {name}: applied {n} Name config label(s) "
                  f"({int(append.sum())} appended to bare ids, "
                  f"{int(replace.sum())} substituted for _unknown)")

    out["_source_file"] = name
    out["_source_row"]  = df.index
    out["_score_col"]   = score_col

    # Drop NaN-score and NaN-model rows
    # A blank `Model version` with a VALID score means the test-taker has no
    # versioned Epoch id, not that the measurement failed. Those rows are still
    # dropped (there is nothing to key a test-taker on), but they are counted and
    # reported: 413 of 4,424 FILE_SPEC rows, ~9% of the Epoch table, and they are
    # a MIX of real base models absent from Epoch's registry (gpt-3.5-turbo-0301,
    # Jurassic-2 Jumbo, Palmyra X), agent scaffolds carrying a step budget
    # (UI-TARS-2-2509 (100 steps)), and task-specific fine-tunes that the project
    # drops on purpose (Mutimodal-T-SciQ_Large). Recovering the first group needs a
    # curated Name -> model_version map; the identity column is captured here so
    # the report can show what is being lost.
    before = len(out)
    ident = "Name" if "Name" in df.columns else ("Agent" if "Agent" in df.columns else None)
    lost = out[out["model_version"].isna() & out["score"].notna()]
    if len(lost):
        labels = (df.loc[lost.index, ident].dropna().astype(str).tolist()
                  if ident else [])
        nan_dropped.append({"file": name, "benchmark": bench_name,
                            "n": int(len(lost)), "labels": labels})
    out = out.dropna(subset=["model_version", "score"])
    if before != len(out):
        print(f"  + {name}: {len(out):,} rows ({before - len(out)} dropped for NaN)")
    else:
        print(f"  + {name}: {len(out):,} rows")
    frames.append(out)

if nan_dropped:
    tot = sum(d["n"] for d in nan_dropped)
    print(f"\nRows dropped for a blank 'Model version' (valid score, no versioned id): {tot}")
    for d in sorted(nan_dropped, key=lambda d: -d["n"])[:8]:
        print(f"  ~ {d['file']}: {d['n']} ({d['benchmark']})")

if unrecognized:
    print(f"\nUnrecognized files (not in FILE_SPEC or FILE_SKIP) — add them or skip them:")
    for n in unrecognized:
        print(f"  ? {n}")

if not frames:
    raise RuntimeError("No frames parsed — check FILE_SPEC.")

scores_long = pd.concat(frames, ignore_index=True, sort=False)
print(f"\nConcatenated: {len(scores_long):,} rows across {scores_long['benchmark'].nunique()} benchmarks")

# Range assertion
out_of_range = scores_long[(scores_long["score"] < 0) | (scores_long["score"] > 1)]
if len(out_of_range):
    print(f"\nFAIL: {len(out_of_range)} scores outside [0, 1]. Sample:")
    print(out_of_range[["model_version", "benchmark", "score", "_source_file"]].head(20).to_string(index=False))
    raise ValueError(
        "Some scores fall outside [0, 1]. Check the divisor in FILE_SPEC for the offending benchmark."
    )

print(f"\nAll {len(scores_long):,} scores in [0, 1].")

out_path = INTERMEDIATE_DIR / "02_scores_long.csv"
scores_long.to_csv(out_path, index=False)
print(f"Wrote {out_path}")


  - SKIP eci_benchmark_difficulties_and_slopes.csv: Epoch's own fitted IRT parameters (downstream of this pipeline)
  + adversarial_nli_external.csv: 15 rows (4 dropped for NaN)
  + aider_polyglot_external.csv: 72 rows (5 dropped for NaN)
  - SKIP ale_bench_external.csv: Performance is a competition points/Elo score (~138-2041), not [0,1]
    · algotune_external.csv: rescaled 'Score' -> 1 - 1/speedup
    · algotune_external.csv: applied 10 Name config label(s) (6 appended to bare ids, 4 substituted for _unknown)
  + algotune_external.csv: 18 rows
    · apex_agents_external.csv: applied 2 Name config label(s) (0 appended to bare ids, 2 substituted for _unknown)
  + apex_agents_external.csv: 55 rows
    · arc_agi_2_external.csv: applied 5 Name config label(s) (5 appended to bare ids, 0 substituted for _unknown)
  + arc_agi_2_external.csv: 172 rows
    · arc_agi_external.csv: applied 6 Name config label(s) (6 appended to bare ids, 0 substituted for _unknown)
  + arc_agi_external.csv: 188 

Wrote /Users/yassineessifi/Desktop/ECI_Bayesian/1_data/1_pipeline/intermediate/02_scores_long.csv


## 03b — RAND biology & chemistry

Epoch's ZIP doesn't cover the biology/chemistry safety-relevant suite
(WMDP, LAB-Bench, MMLU Pro Biology/Chemistry, BioLP-bench, GPQA Diamond
Biology/Chemistry, plus the GPQA Main and MMLU Biology/Chemistry
sub-splits). The upstream-of-record is:

> Dev et al. (2025), *Toward Comprehensive Benchmarking of the Biological
> Knowledge of Frontier Large Language Models*, RAND Corporation
> techreport **RR-A3797-1** —
> https://www.rand.org/pubs/research_reports/RRA3797-1.html

RAND ships the data as PDF tables inside the report rather than a
machine-readable CSV. The General-Purpose-AI-Policy-Lab "benchmark-
forecasting" repo published a digitized copy. This section pulls that
file (cached into the snapshot folder for provenance) and reshapes it
wide → long.

The CSV's first two rows hold human baselines (per-benchmark score +
description). They are skipped, not ingested: `1_data/curated/human_baselines.csv`
is the authoritative human file, hand-filed by tier. A blanket tier label here
mis-filed the MMLU splits (RAND's baseline for them is a 95th-percentile
test-taker, not an in-domain PhD) and double-counted one measurement.


In [5]:
# Original source-of-record: RAND techreport (PDF tables).
# We pull the digitized CSV maintained by the General-Purpose-AI-Policy-Lab.
RAND_URL = (
    "https://raw.githubusercontent.com/General-Purpose-AI-Policy-Lab/"
    "benchmark-forecasting/main/Data/benchmark_data_RAND/benchmark_scores_RAND.csv"
)
RAND_SOURCE_LABEL = "RAND (Dev et al. 2025) — RR-A3797-1"
RAND_SOURCE_LINK  = "https://www.rand.org/pubs/research_reports/RRA3797-1.html"

# Wide-format column → canonical benchmark name
RAND_BENCH_MAP = {
    "Bio: BioLP-bench":             "BioLP-bench",
    "Bio: LAB-Bench Cloning":       "LAB-Bench Cloning",
    "Bio: LAB-Bench LitQA2":        "LAB-Bench LitQA2",
    "Bio: LAB-Bench SeqQA":         "LAB-Bench SeqQA",
    "Bio: LAB-Bench Protocol":      "LAB-Bench Protocol",
    "Bio: GPQA Diamond Biology":    "GPQA Diamond Biology",
    "Bio: GPQA Main Biology":       "GPQA Main Biology",
    "Bio: MMLU Biology":            "MMLU Biology",
    "Bio: WMDP Biology":            "WMDP Biology",
    "Bio: MMLU Pro Biology":        "MMLU Pro Biology",
    "Chem: GPQA Main Chemistry":    "GPQA Main Chemistry",
    "Chem: GPQA Diamond Chemistry": "GPQA Diamond Chemistry",
    "Chem: MMLU Chemistry":         "MMLU Chemistry",
    "Chem: WMDP Chemistry":         "WMDP Chemistry",
    "Chem: MMLU Pro Chemistry":     "MMLU Pro Chemistry",
}

rand_path = LATEST_SNAPSHOT / "rand_benchmark_scores.csv"
if not rand_path.exists():
    print(f"Downloading {RAND_URL}")
    urllib.request.urlretrieve(RAND_URL, rand_path)
    print(f"  -> {rand_path}")
else:
    print(f"RAND CSV already in snapshot — re-using.")

rand_raw = pd.read_csv(rand_path)

# First two rows are RAND's human baselines (score + description). Skipped:
# 1_data/curated/human_baselines.csv is the authoritative human file, hand-filed
# by tier. RAND's MMLU baselines are 95th-percentile test-takers, not in-domain
# PhDs, so the blanket "Domain Expert" label used here double-counted them.
rand_models = rand_raw.iloc[2:].copy()

# Wide → long, restrict to our mapped columns, scale 0-100 -> 0-1
rand_long = rand_models.melt(id_vars=["Model"], var_name="bench_col", value_name="score_raw")
rand_long = rand_long[rand_long["bench_col"].isin(RAND_BENCH_MAP)].copy()
rand_long["benchmark"]     = rand_long["bench_col"].map(RAND_BENCH_MAP)
rand_long["score"]         = pd.to_numeric(rand_long["score_raw"], errors="coerce") / 100.0
rand_long["model_version"] = rand_long["Model"]
rand_long["release_date"]  = pd.NaT
rand_long["organization"]  = ""
rand_long["source"]        = RAND_SOURCE_LABEL
rand_long["source_link"]   = RAND_SOURCE_LINK
rand_long["_source_file"]  = "rand_benchmark_scores.csv"
rand_long["_source_row"]   = rand_long.index
rand_long["_score_col"]    = rand_long["bench_col"]
rand_long = rand_long.dropna(subset=["score"])

keep_cols = ["model_version", "score", "benchmark", "release_date", "organization",
             "source", "source_link", "_source_file", "_source_row", "_score_col"]
rand_long = rand_long[keep_cols]

# Range check
oor = rand_long[(rand_long["score"] < 0) | (rand_long["score"] > 1)]
if len(oor):
    raise ValueError(f"{len(oor)} RAND scores outside [0,1] — check the /100 divisor")

print(f"\nRAND model scores: {len(rand_long):,} rows × {rand_long['benchmark'].nunique()} benchmarks × {rand_long['model_version'].nunique()} models")
print(f"Score range: {rand_long['score'].min():.3f} – {rand_long['score'].max():.3f}")

# Append to scores_long produced in section 03
scores_long = pd.concat([scores_long, rand_long], ignore_index=True, sort=False)
print(f"Total scores_long after RAND merge: {len(scores_long):,} rows × {scores_long['benchmark'].nunique()} benchmarks")

# Re-write the intermediate CSV with the appended RAND rows
scores_long.to_csv(INTERMEDIATE_DIR / "02_scores_long.csv", index=False)


  -> /Users/yassineessifi/Desktop/ECI_Bayesian/1_data/1_pipeline/snapshots/2026-08-07/rand_benchmark_scores.csv

RAND model scores: 565 rows × 15 benchmarks × 40 models
Score range: 0.010 – 0.922
Total scores_long after RAND merge: 4,250 rows × 74 benchmarks

RAND human baselines extracted: 13 rows (for section 08)
             benchmark         group  score                         note                                                    source
           BioLP-bench Domain Expert  0.384 Bachelor's w/ lab experience https://www.rand.org/pubs/research_reports/RRA3797-1.html
     LAB-Bench Cloning Domain Expert  0.600                In-domain PhD https://www.rand.org/pubs/research_reports/RRA3797-1.html
      LAB-Bench LitQA2 Domain Expert  0.700                In-domain PhD https://www.rand.org/pubs/research_reports/RRA3797-1.html
       LAB-Bench SeqQA Domain Expert  0.780                In-domain PhD https://www.rand.org/pubs/research_reports/RRA3797-1.html
    LAB-Bench Protocol Domai

## 03c — Scale SEAL leaderboards

Scale's [SEAL leaderboards](https://labs.scale.com/leaderboard) rank frontier
models on private, expert-graded datasets. They cover capability axes the Epoch
suite barely touches (multi-turn instruction following, tool use, multilingual
reasoning) and they populate benchmark stubs already present in
`canonical/benchmark_metadata.csv` (EnigmaEval, MCP Atlas, MultiChallenge,
MultiNRC, VISTA, TutorBench, SWE-Bench Pro).

There is **no public CSV/JSON download and no documented API**. The leaderboard
pages are Next.js apps that server-render the data into `self.__next_f` streaming
chunks; the visible `<table>` is a stale snapshot, so we parse the chunks. Each
board exposes an `entries` array of `{model, score, confidenceInterval_upper,
company, createdAt, deprecated, ...}` with scores on 0–100.

`SEAL_SPEC` maps `slug → (canonical_benchmark_name, divisor, status)`. `SEAL_SKIP`
records boards we deliberately omit and why (Elo-scaled `coding` and the
multilingual rating boards; the text-only HLE that would collide with Epoch's full
HLE). Slugs come from Scale's Sanity CMS (some use hyphens / short forms, e.g.
`prbench-finance`, `vtb`, `rli`, `audiomc`); every board server-renders its data
under the correct slug. Raw HTML per slug is
cached under `snapshots/<date>/seal/` for reproducibility, alongside
`seal_provenance.json`.

> **Licensing:** unlike Epoch's CC-BY ZIP, SEAL data carries no open license. It
> is aggregated here as fair-use research with attribution to Scale AI; confirm
> terms before publishing derived analysis.

The `confidenceInterval_upper` (± CI) is **not** captured — the long-frame schema
carries no `stderr` column downstream (Epoch's measured stderr is likewise
dropped today). Wiring stderr end-to-end across all sources is a separate
follow-up. `createdAt` is the leaderboard-add date, not a model release date, so
`release_date` is left null (`analysis.py` takes the earliest release date per
model, picking up the true Epoch date for any overlapping model).


In [6]:
import requests

# Scale SEAL leaderboards (labs.scale.com/leaderboard/<slug>). No public CSV/JSON
# download and no documented API. The live data is server-rendered into the page as
# Next.js streaming chunks (`self.__next_f.push([...])`); the visible <table> markup is
# a STALE snapshot, so we parse the chunks, not the table. Each leaderboard carries an
# `entries` array of {model, score, confidenceInterval_upper, company, createdAt,
# deprecated, ...}. Scores are 0-100 -> /100.
SEAL_BASE         = "https://labs.scale.com/leaderboard/"
SEAL_SOURCE_LABEL = "Scale SEAL"

# slug source: Scale's leaderboard list lives in their Sanity CMS, queryable via
# https://5uhyv5jy.apicdn.sanity.io/v2022-03-07/data/query/production
#   ?query=*[_type=="leaderboard"]{"slug":slug.current,title,scoreTitle}
# Use the canonical slug.current values (note hyphens: prbench-finance; short
# forms: vtb, rli, audiomc). Every board SSRs its data under the correct slug.
# slug -> (canonical_benchmark_name, divisor, status). status="deprecated" only
# downgrades a 404 from warning to info; the data (if present) is still ingested.
SEAL_SPEC = {
    # enigma_eval intentionally absent: the board stopped serving its table to
    # this scraper (47 rows -> 5 on 2026-07-27) and Epoch now ships the full
    # 76-row file in the ZIP. Ingested via FILE_SPEC in section 03 instead.
    "mcp_atlas":                     ("MCP Atlas",                      100.0, "active"),
    "multichallenge":                ("MultiChallenge",                 100.0, "active"),
    "multinrc":                      ("MultiNRC",                       100.0, "active"),
    "visual_language_understanding": ("Visual Task Assessment (VISTA)", 100.0, "active"),
    "tutorbench":                    ("TutorBench",                     100.0, "active"),
    "tool_use_enterprise":           ("SEAL Tool Use (Enterprise)",     100.0, "active"),
    "swe_bench_pro_public":          ("SWE-Bench Pro",                  100.0, "active"),
    "swe_bench_pro_private":         ("SWE-Bench Pro (Private)",        100.0, "active"),
    "instruction_following":         ("SEAL Instruction Following",     100.0, "deprecated"),
    # --- orphan stubs filled (correct Sanity slugs) ---
    "prbench-finance":               ("PRBench Finance",               100.0, "active"),
    "prbench-legal":                 ("PRBench Legal",                 100.0, "active"),
    # rli is ALSO in the ZIP (rli_external.csv), and both are kept on purpose. It is
    # a PARTIAL overlap, not a duplicate: the ZIP carries 10 models, this board 15,
    # union 17. Dropping the scrape lost 7 models (measured 2026-07-28), so the
    # "prefer the ZIP" rule does not apply here. The 2 shared rows disagree by 0.003
    # (claude-fable-5 0.158 vs 0.161) and dedup takes the max.
    "rli":                           ("Remote Labor Index",            100.0, "active"),
    "vtb":                           ("VisualToolBench",               100.0, "active"),
    "audiomc":                       ("AudioMultiChallenge",           100.0, "active"),
}

# Slugs deliberately not ingested, with reason (mirrors FILE_SKIP). Every SEAL board
# server-renders its data under its correct Sanity slug -- the omissions below are by
# score-scale / comparability, NOT fetchability. Other reachable-but-omitted boards
# (fortress, mask, propensitybench, math, tool_use_chat, sweatlas-*, scipredict, hil,
# korean) are left out of the canonical broad index for now -- see the roadmap.
SEAL_SKIP = {
    "humanitys_last_exam_text_only": "text-only subset; not comparable to Epoch's full HLE (user decision)",
    "coding":                        "Elo-style rating (~600-1240), not a [0,1] score",
    "arabic":                        "multilingual Elo-style rating, not a [0,1] score",
    "chinese":                       "multilingual Elo-style rating, not a [0,1] score",
    "japanese":                      "multilingual Elo-style rating, not a [0,1] score",
    "spanish":                       "multilingual Elo-style rating, not a [0,1] score",
}


SEAL_HEADERS = {"User-Agent": "Mozilla/5.0"}


def _seal_decode_chunks(html):
    """Concatenate + JSON-unescape the self.__next_f.push streaming chunks.

    Per-chunk json.loads handles the \\uXXXX / \\n / \\" escapes AND preserves
    literal UTF-8 (e.g. the U+2020 dagger footnote marker). A plain
    .decode("unicode_escape") treats the bytes as Latin-1 and mangles every
    multi-byte char into mojibake (dagger -> 'a-circumflex + C1 controls')."""
    out = []
    for chunk in re.findall(r'self\.__next_f\.push\(\[\d+,\s*"((?:[^"\\]|\\.)*)"\]\)', html):
        try:
            out.append(json.loads('"' + chunk + '"'))
        except Exception:
            out.append(chunk.encode("latin-1", "backslashreplace").decode("unicode_escape", "replace"))
    return "".join(out)


def _seal_extract_entries(decoded):
    """Return the largest `"entries":[...]` array whose items have a `model` key."""
    arrays = []
    for m in re.finditer(r'"entries"\s*:\s*\[', decoded):
        start = m.end() - 1  # at the '['
        depth, i = 0, start
        while i < len(decoded):
            c = decoded[i]
            if c == "[":
                depth += 1
            elif c == "]":
                depth -= 1
                if depth == 0:
                    break
            i += 1
        try:
            arr = json.loads(decoded[start:i + 1])
        except Exception:
            continue
        if arr and isinstance(arr[0], dict) and "model" in arr[0]:
            arrays.append(arr)
    return max(arrays, key=len) if arrays else None


seal_dir = LATEST_SNAPSHOT / "seal"
seal_dir.mkdir(parents=True, exist_ok=True)

seal_frames, seal_prov_slugs = [], []

# Re-run safety: only scrape/append if SEAL rows aren't already in scores_long.
if (scores_long["source"] == SEAL_SOURCE_LABEL).any():
    print("Scale SEAL rows already present in scores_long — skipping re-fetch "
          "(Restart & Run All for a clean rebuild).")
else:
    for slug, (bench_name, divisor, status) in SEAL_SPEC.items():
        url   = SEAL_BASE + slug
        cache = seal_dir / f"{slug}.html"
        http_status, parse_method = None, "skipped"
        try:
            if cache.exists():
                html, http_status = cache.read_text(), 200
            else:
                resp = requests.get(url, headers=SEAL_HEADERS, timeout=30)
                http_status = resp.status_code
                if http_status != 200:
                    lvl = "info" if status == "deprecated" else "WARN"
                    print(f"  [{lvl}] SEAL {slug}: HTTP {http_status} — skipping")
                    seal_prov_slugs.append({"slug": slug, "url": url, "status": status,
                                            "http_status": http_status, "sha256": None,
                                            "byte_size": 0, "n_rows": 0, "parse_method": "http_error"})
                    continue
                html = resp.text
                cache.write_text(html)
        except Exception as e:
            print(f"  WARN SEAL {slug}: fetch failed ({e}) — skipping")
            seal_prov_slugs.append({"slug": slug, "url": url, "status": status,
                                    "http_status": http_status, "sha256": None,
                                    "byte_size": 0, "n_rows": 0, "parse_method": "fetch_error"})
            continue

        entries = _seal_extract_entries(_seal_decode_chunks(html))
        if not entries:
            print(f"  WARN SEAL {slug}: no entries array in SSR payload — skipping "
                  f"(manual capture: drop seal/{slug}.csv into the snapshot)")
            seal_prov_slugs.append({"slug": slug, "url": url, "status": status,
                                    "http_status": http_status, "sha256": None,
                                    "byte_size": len(html), "n_rows": 0, "parse_method": "no_entries"})
            continue

        edf = pd.DataFrame(entries)
        org = edf["company"].fillna("") if "company" in edf.columns else ""
        sub = pd.DataFrame({
            "model_version": edf["model"],
            "score":         pd.to_numeric(edf["score"], errors="coerce") / divisor,
            "benchmark":     bench_name,
            "release_date":  pd.NaT,   # SEAL `createdAt` is the leaderboard-add date, not
                                       # a model release date — leave unknown (mirrors RAND).
            "organization":  org,
            "source":        SEAL_SOURCE_LABEL,
            "source_link":   url,
            "_source_file":  f"seal/{slug}.html",
            "_source_row":   edf.index,
            "_score_col":    "score",
        })
        sub = sub.dropna(subset=["model_version", "score"])
        seal_frames.append(sub)
        seal_prov_slugs.append({"slug": slug, "url": url, "status": status,
                                "http_status": http_status,
                                "sha256": hashlib.sha256(html.encode()).hexdigest(),
                                "byte_size": len(html), "n_rows": int(len(sub)),
                                "parse_method": "next_f_entries"})
        print(f"  + SEAL {slug}: {len(sub):,} rows -> {bench_name}")

    for slug, reason in SEAL_SKIP.items():
        print(f"  - SKIP SEAL {slug}: {reason}")

    if seal_frames:
        seal_long = pd.concat(seal_frames, ignore_index=True, sort=False)

        oor = seal_long[(seal_long["score"] < 0) | (seal_long["score"] > 1)]
        if len(oor):
            bad = sorted(oor["benchmark"].unique())
            raise ValueError(f"SEAL scores outside [0,1] for {bad} — check the divisor in SEAL_SPEC")

        print(f"\nSEAL scores: {len(seal_long):,} rows x {seal_long['benchmark'].nunique()} "
              f"benchmarks x {seal_long['model_version'].nunique()} models "
              f"(range {seal_long['score'].min():.3f}-{seal_long['score'].max():.3f})")

        scores_long = pd.concat([scores_long, seal_long], ignore_index=True, sort=False)
        scores_long.to_csv(INTERMEDIATE_DIR / "02_scores_long.csv", index=False)
        print(f"Total scores_long after SEAL merge: {len(scores_long):,} rows x "
              f"{scores_long['benchmark'].nunique()} benchmarks")
    else:
        print("\nNo SEAL frames parsed — scores_long unchanged.")

    seal_prov = {
        "source":      SEAL_SOURCE_LABEL,
        "fetched_at":  datetime.now(timezone.utc).isoformat(),
        "base_url":    SEAL_BASE,
        "license":     "Scale AI ToS — not CC-BY; used as fair-use research aggregation, attribute Scale AI",
        "attribution": "Scale AI — SEAL leaderboards",
        "slugs":       seal_prov_slugs,
    }
    (seal_dir / "seal_provenance.json").write_text(json.dumps(seal_prov, indent=2))
    print(f"Wrote {seal_dir / 'seal_provenance.json'}")


  + SEAL mcp_atlas: 30 rows -> MCP Atlas


  + SEAL multichallenge: 30 rows -> MultiChallenge


  + SEAL multinrc: 44 rows -> MultiNRC


  + SEAL visual_language_understanding: 63 rows -> Visual Task Assessment (VISTA)


  + SEAL tutorbench: 27 rows -> TutorBench


...

  + SEAL prbench-finance: 31 rows -> PRBench Finance


  + SEAL prbench-legal: 31 rows -> PRBench Legal


  + SEAL rli: 15 rows -> Remote Labor Index


  + SEAL vtb: 21 rows -> VisualToolBench


  + SEAL audiomc: 33 rows -> AudioMultiChallenge
  - SKIP SEAL humanitys_last_exam_text_only: text-only subset; not comparable to Epoch's full HLE (user decision)
  - SKIP SEAL coding: Elo-style rating (~600-1240), not a [0,1] score
  - SKIP SEAL arabic: multilingual Elo-style rating, not a [0,1] score
  - SKIP SEAL chinese: multilingual Elo-style rating, not a [0,1] score
  - SKIP SEAL japanese: multilingual Elo-style rating, not a [0,1] score
  - SKIP SEAL spanish: multilingual Elo-style rating, not a [0,1] score

SEAL scores: 418 rows x 14 benchmarks x 271 models (range 0.008-0.920)
Total scores_long after SEAL merge: 4,668 rows x 87 benchmarks
Wrote /Users/yassineessifi/Desktop/ECI_Bayesian/1_data/1_pipeline/snapshots/2026-08-07/seal/seal_provenance.json


## 03d — FrontierMath v2 (live feed)

The two FrontierMath files in the ZIP carry only pre-2026-06-12 runs (the v1
problem set). Epoch's `benchmarks.csv` feed carries both series and tags them,
so the corrected v2 measurements can be selected explicitly.

`task version` identifies the problem set; `task` distinguishes the private
evaluation sets from the twelve public problems. Private only, matching the
convention the ZIP files followed.

The canonical benchmark names are kept (`FrontierMath`, `FrontierMath Tier 4`)
so the curated floors, the exclusion list, and every downstream figure keep
resolving. What changes is which problem set those names refer to.


In [7]:
# Epoch internal evals taken from the live feed rather than the ZIP, either
# because the ZIP lags a problem-set version (FrontierMath) or because the ZIP
# ships no file for the benchmark at all (EBR-bench).
# live task name -> (canonical benchmark name, required `task version` prefix or None)
LIVE_TASK_SPEC = {
    "FrontierMath-Tiers-1-3-v2-Private": ("FrontierMath",         "2."),
    "FrontierMath-Tier-4-v2-Private":    ("FrontierMath Tier 4",  "2."),
    # v1 kept as SEPARATE benchmarks, the way Epoch itself reports the two series.
    # Not pooled with v2: the 2026-06-12 rerun fixed errors in 42% of problems and
    # moved the mean from 0.18 to 0.59, so one column would carry a bimodal
    # difficulty. As its own item each version gets its own D, A and sigma_b, so a
    # defective instrument reads as a low-discrimination noisy one rather than
    # contaminating the v2 scale. `task version` here is 1.x harness revisions on
    # one problem set, so all 1.* pool. Public splits stay skipped below.
    "FrontierMath-2025-02-28-Private":        ("FrontierMath v1",        "1."),
    "FrontierMath-Tier-4-2025-07-01-Private": ("FrontierMath Tier 4 v1", "1."),
    # No ale_bench-style ZIP file exists for this one: it is on the site and in
    # the live feed only. Unversioned upstream, so no version pin.
    "EBR-bench":                         ("EBR-bench",            None),
    # Also in the ZIP, byte-identically; routed here so all internal evals share one
    # transport. No version pin: `task version` is a HARNESS version for these
    # (Chess Puzzles has 8 distinct values, GPQA 12), not a problem-set marker.
    "Chess Puzzles":                     ("Chess Puzzles",             None),
    "GPQA diamond":                      ("GPQA Diamond",              None),
    "MATH level 5":                      ("MATH Level 5",              None),
    "OTIS Mock AIME 2024-2025":          ("OTIS Mock AIME 2024-2025",  None),
    "SWE-Bench verified":                ("SWE-Bench Verified",        None),
    "SimpleQA Verified":                 ("SimpleQA Verified",         None),
    # Epoch internal eval, first seen 2026-08-04. Carries stderr like the other
    # internal evals, so it lands with a usable instrument precision. Unversioned
    # upstream, so no problem-set pin.
    "Mystery Game Puzzles":              ("Mystery Game Puzzles",      None),
}
LIVE_SOURCE_LABEL = "EpochAI (live feed)"

# Tasks in the feed we deliberately do NOT ingest, with the reason — same role as
# FILE_SKIP in section 03. Anything in the feed that is in neither dict is NEW
# upstream data and gets reported, so a task cannot appear unnoticed the way seven
# ZIP benchmarks did.
LIVE_TASK_SKIP = {
    # Public splits are 10 and 2 problems: 42/87 and 46/48 rows sit exactly on a
    # boundary, so most of their information would be manufactured by the clip.
    "FrontierMath-2025-02-28-Public":         "10-problem public split; boundary-degenerate",
    "FrontierMath-Tier-4-2025-07-01-Public":  "2-problem public split; only 3 distinct values",
}

live = pd.read_csv(LATEST_LIVE_CSV, low_memory=False)
print(f"Live feed: {len(live):,} rows x {live['task'].nunique()} tasks")

live_unrecognized = sorted(set(live["task"].dropna().unique())
                           - set(LIVE_TASK_SPEC) - set(LIVE_TASK_SKIP))
if live_unrecognized:
    print(f"\nUnrecognized live-feed tasks (in neither LIVE_TASK_SPEC nor LIVE_TASK_SKIP):")
    for t in live_unrecognized:
        print(f"  ? {t}")

missing = sorted(set(LIVE_TASK_SPEC) - set(live["task"].unique()))
if missing:
    raise ValueError(
        f"Expected v2 task(s) absent from the live feed: {missing}. "
        f"Available FrontierMath tasks: "
        f"{sorted(t for t in live['task'].unique() if 'FrontierMath' in str(t))}"
    )

live_frames = []
for task, (bench_name, ver_prefix) in LIVE_TASK_SPEC.items():
    sub = live[live["task"] == task].copy()

    # Guard the version selection: a silent upstream retag would otherwise pull
    # a different problem set in under the same task name.
    ver = sub["task version"].astype(str)
    # A few runs carry no version at all (4 per v1 task). The task NAME already
    # pins the problem set, since v2 runs live under their own task names, so the
    # prefix is a second line of defence against an unannounced problem-set bump
    # rather than the primary selector. Unversioned rows are kept and checked
    # only for the versions they do report.
    seen = sub["task version"].notna()   # astype(str) leaves NA as NA, not "nan"
    if ver_prefix is not None and not ver[seen].str.startswith(ver_prefix).all():
        bad = sorted(ver[seen][~ver[seen].str.startswith(ver_prefix)].unique())
        raise ValueError(f"{task}: unexpected task version(s) {bad}")

    out = pd.DataFrame({
        "model_version": sub["model"],
        "score":         pd.to_numeric(sub["mean_score"], errors="coerce"),
        "benchmark":     bench_name,
        "release_date":  sub["Version release date"],
        "organization":  sub["Organization"],
        "stderr":        pd.to_numeric(sub["stderr"], errors="coerce"),
        "source":        LIVE_SOURCE_LABEL,
    })
    out["_source_file"] = "epoch_live_benchmarks.csv"
    out["_source_row"]  = sub.index
    out["_score_col"]   = "mean_score"

    n_unver = int((~seen).sum())
    fmt_ver = (f"task version {sorted(ver[seen].unique())}"
               + (f" +{n_unver} unversioned" if n_unver else "")) if ver_prefix else (
               f"{ver.nunique()} harness version(s), unpinned")
    before = len(out)
    out = out.dropna(subset=["model_version", "score"])
    print(f"  + {task} -> {bench_name!r}: {len(out):,} rows"
          f"{f' ({before - len(out)} dropped for NaN)' if before != len(out) else ''}"
          f"  [{fmt_ver}]")
    live_frames.append(out)

fm_long = pd.concat(live_frames, ignore_index=True, sort=False)

oor = fm_long[(fm_long["score"] < 0) | (fm_long["score"] > 1)]
if len(oor):
    raise ValueError(f"{len(oor)} live-feed scores outside [0,1]")

# The ZIP's v1 rows are skipped in 03, so nothing should already claim these names.
live_names = [b for b, _ in LIVE_TASK_SPEC.values()]
clash = scores_long[scores_long["benchmark"].isin(live_names)]
if len(clash):
    raise ValueError(
        f"{len(clash)} pre-existing rows on {sorted(clash['benchmark'].unique())} — "
        "the v1 FILE_SPEC entries should be in FILE_SKIP (section 03)."
    )

print(f"\nLive feed: {len(fm_long):,} rows x {fm_long['benchmark'].nunique()} "
      f"benchmarks x {fm_long['model_version'].nunique()} models "
      f"(range {fm_long['score'].min():.3f}-{fm_long['score'].max():.3f})")

scores_long = pd.concat([scores_long, fm_long], ignore_index=True, sort=False)
scores_long.to_csv(INTERMEDIATE_DIR / "02_scores_long.csv", index=False)
print(f"Total scores_long after live-feed merge: {len(scores_long):,} rows x "
      f"{scores_long['benchmark'].nunique()} benchmarks")


Live feed: 1,283 rows x 14 tasks
  + FrontierMath-Tiers-1-3-v2-Private -> 'FrontierMath': 42 rows  [task version ['2.0.0']]
  + FrontierMath-Tier-4-v2-Private -> 'FrontierMath Tier 4': 44 rows  [task version ['2.0.0']]
  + FrontierMath-2025-02-28-Private -> 'FrontierMath v1': 101 rows  [task version ['1.0.0', '1.0.1', '1.1.0', '1.1.1', '1.1.2', '1.1.4', '1.1.5', '1.1.6', '1.1.8', '1.1.9'] +4 unversioned]
  + FrontierMath-Tier-4-2025-07-01-Private -> 'FrontierMath Tier 4 v1': 72 rows  [task version ['1.0.0', '1.0.1', '1.1.0', '1.1.1', '1.1.2', '1.1.3', '1.1.4', '1.1.5', '1.1.6', '1.1.8', '1.1.9'] +4 unversioned]
  + EBR-bench -> 'EBR-bench': 18 rows  [1 harness version(s), unpinned]
  + Chess Puzzles -> 'Chess Puzzles': 144 rows  [8 harness version(s), unpinned]
  + GPQA diamond -> 'GPQA Diamond': 246 rows  [12 harness version(s), unpinned]
  + MATH level 5 -> 'MATH Level 5': 108 rows  [2 harness version(s), unpinned]
  + OTIS Mock AIME 2024-2025 -> 'OTIS Mock AIME 2024-2025': 221 rows 

## 03e — Kaggle Open Benchmarks (JSON API)

Kaggle's "Open Benchmarks" boards (`kaggle.com/benchmarks/<owner>/<slug>`) are
read via the same JSON endpoint the pages themselves call — no API key, no
`kaggle` SDK. Three boards are ingested, each a leaderboard Epoch does not
carry: `open-benchmarks/mmlu`, `open-benchmarks/mmlu-pro`,
`deepmind/simpleqa-verified`.

Kaggle is the sole source for MMLU-Pro. MMLU and SimpleQA Verified also have
rows from the ZIP and the live feed, so
this section creates real (model, benchmark) collisions on purpose;
`KAGGLE_SPEC`'s `kaggle_wins` flag decides the winner per benchmark and is read
by the existing dedup step below (section 05) as a source-priority rule, not a
new resolution mechanism.


In [8]:
# Board -> (canonical benchmark, kaggle_wins). kaggle_wins is the merge policy
# for the (model, benchmark) collisions this feed creates against rows already
# in scores_long; it is read by the dedup step in section 05 below, not
# resolved here.
#   MMLU, MMLU-Pro      True  — no curated alternative worth keeping (MMLU-Pro
#                                had none at all; see the note above), so the
#                                Kaggle board wins any overlap outright.
#   SimpleQA Verified    False — the live feed (section 03d) already carries
#                                this benchmark; Kaggle only fills in models
#                                our existing column is missing.
KAGGLE_SPEC = {                              # owner/slug: (benchmark, kaggle_wins)
    "open-benchmarks/mmlu":       ("MMLU",              True),
    "open-benchmarks/mmlu-pro":   ("MMLU-Pro",          True),
    "deepmind/simpleqa-verified": ("SimpleQA Verified", False),
}
KAGGLE_URL = "https://www.kaggle.com/api/v1/benchmarks/{}/leaderboard"
KAGGLE_SOURCE_FMT = "Kaggle Open Benchmarks ({})"

# A board's reported confidence interval must imply its own known item count,
# or it is not the benchmark we think it is. This one check has caught three
# broken boards before ingestion: HELM Lite's MMLU number came from a 5-subject
# subsample rather than the full 57, an AIME 2025 board reported a CI of
# exactly 0 (no binomial noise model behind it), and a personal-account GSM8K
# board's "scores" ran to 65,250 (a raw item count, not a proportion).
# n_eff = p(1-p)/se^2 recovers the effective item count from the reported CI.
KAGGLE_N_ITEMS = {"MMLU": 14042, "MMLU-Pro": 12032, "SimpleQA Verified": 1000}

def _check_kaggle_instruments(k):
    for bench, n in KAGGLE_N_ITEMS.items():
        g = k[k.benchmark == bench]
        neff = (g.score * (1 - g.score) / g.stderr ** 2).median()
        assert 0.9 * n < neff < 1.1 * n, f"{bench}: n_eff {neff:.0f} != {n}"

if scores_long["source"].astype(str).str.startswith("Kaggle Open Benchmarks").any():
    print("Kaggle rows already present in scores_long — skipping.")
else:
    kaggle_dir = LATEST_SNAPSHOT / "kaggle"
    kaggle_dir.mkdir(parents=True, exist_ok=True)

    kaggle_frames = []
    for owner_slug, (bench, kaggle_wins) in KAGGLE_SPEC.items():
        cache_path = kaggle_dir / (owner_slug.replace("/", "_") + ".json")
        if cache_path.exists():
            print(f"Kaggle {owner_slug} already snapshotted at {cache_path.name} — re-using.")
        else:
            url = KAGGLE_URL.format(owner_slug)
            print(f"Downloading {url}")
            urllib.request.urlretrieve(url, cache_path)

        payload = json.loads(cache_path.read_text())
        recs, n_skip = [], 0
        for row in payload.get("rows", []):
            # No effort-suffix convention on this board (unlike the ZIP/live
            # feed's _low/_medium/_high); the only per-model cleanup Kaggle
            # needs is its own "-default" harness-mode tag on the slug.
            model = re.sub(r"-default$", "", row.get("modelVersionSlug", ""))
            for tr in row.get("taskResults", []):
                num = (tr.get("result") or {}).get("numericResult")
                if not num or "value" not in num:
                    n_skip += 1   # e.g. 2 of 53 rows on the MMLU board
                    continue
                ci = num.get("confidenceInterval")   # binomial 95% half-width
                recs.append({
                    "model_version": model,
                    "score":         num["value"],
                    "stderr":        (ci / 1.96) if ci is not None else np.nan,
                })

        sub = pd.DataFrame(recs)
        sub["benchmark"]    = bench
        sub["release_date"] = pd.NaT   # board carries a per-run evaluation date, not a model release date
        sub["organization"] = ""
        sub["source"]       = KAGGLE_SOURCE_FMT.format(owner_slug)
        sub["_source_file"] = cache_path.name
        sub["_score_col"]   = "numericResult.value"
        print(f"  + {owner_slug} -> {bench!r}: {len(sub)} rows "
              f"({n_skip} skipped, no numericResult.value)  [kaggle_wins={kaggle_wins}]")
        kaggle_frames.append(sub)

    kaggle_long = pd.concat(kaggle_frames, ignore_index=True, sort=False)

    oor = kaggle_long[(kaggle_long["score"] < 0) | (kaggle_long["score"] > 1)]
    if len(oor):
        raise ValueError(f"{len(oor)} Kaggle scores outside [0,1]")

    _check_kaggle_instruments(kaggle_long)
    print("Instrument check passed: each board's CI implies its known item count.")

    # Unlike 03d, a clash with existing scores_long rows is EXPECTED here — it is
    # exactly what kaggle_wins resolves in section 05 below — so it is not guarded
    # against the way the live-feed section guards it.
    scores_long = pd.concat([scores_long, kaggle_long], ignore_index=True, sort=False)
    print(f"\nKaggle Open Benchmarks: {len(kaggle_long):,} rows x "
          f"{kaggle_long['benchmark'].nunique()} benchmarks x "
          f"{kaggle_long['model_version'].nunique()} models")
    print(f"Total scores_long after Kaggle merge: {len(scores_long):,} rows x "
          f"{scores_long['benchmark'].nunique()} benchmarks")


  + open-benchmarks/mmlu -> 'MMLU': 51 rows (2 skipped, no numericResult.value)  [kaggle_wins=True]


  + open-benchmarks/mmlu-pro -> 'MMLU-Pro': 55 rows (0 skipped, no numericResult.value)  [kaggle_wins=True]


  + deepmind/simpleqa-verified -> 'SimpleQA Verified': 46 rows (1 skipped, no numericResult.value)  [kaggle_wins=False]
Instrument check passed: each board's CI implies its known item count.

Kaggle Open Benchmarks: 152 rows x 3 benchmarks x 67 models
Total scores_long after Kaggle merge: 5,968 rows x 100 benchmarks


## 04 — Canonicalize names

Apply `canonical/benchmark_names.csv` and `canonical/model_aliases.csv`
(both `variant → canonical → note`). The alias file contains only **true
renames** and DROP rows — identity mappings (model already has its canonical
name) live in `canonical/reviewed_models.txt` (one name per line).

Every rename gets recorded in `output/name_changes.csv`. Models that
have no row in `model_aliases.csv` and aren't listed in
`reviewed_models.txt` (= new since the last canonicalization review)
get flagged in `output/missing_models.csv`.


In [9]:
NAMES_CSV    = CANONICAL_DIR / "benchmark_names.csv"
ALIASES_CSV  = CANONICAL_DIR / "model_aliases.csv"
REVIEWED_TXT = CANONICAL_DIR / "reviewed_models.txt"

# Auto-seed on first run from observed variants
if not NAMES_CSV.exists():
    seed = (scores_long[["benchmark"]].drop_duplicates().sort_values("benchmark")
            .rename(columns={"benchmark": "variant"}))
    seed["canonical"] = seed["variant"]
    seed["note"]      = ""
    seed.to_csv(NAMES_CSV, index=False)
    print(f"Seeded {NAMES_CSV.name} with {len(seed)} identity rows.")

if not ALIASES_CSV.exists():
    pd.DataFrame(columns=["variant", "canonical", "note"]).to_csv(ALIASES_CSV, index=False)
    print("Seeded empty model_aliases.csv (true renames only).")

if not REVIEWED_TXT.exists():
    # Seed with every model name currently in the data
    reviewed = sorted(scores_long["model_version"].unique())
    REVIEWED_TXT.write_text("\n".join(reviewed) + "\n")
    print(f"Seeded {REVIEWED_TXT.name} with {len(reviewed)} model names.")

bench_df = pd.read_csv(NAMES_CSV)
alias_df = pd.read_csv(ALIASES_CSV)
bench_map = dict(zip(bench_df["variant"], bench_df["canonical"]))
model_map = dict(zip(alias_df["variant"], alias_df["canonical"]))
reviewed_models = set(REVIEWED_TXT.read_text().splitlines())

named = scores_long.copy()
named["benchmark_raw"]     = named["benchmark"]
named["model_version_raw"] = named["model_version"]
# Strip leaderboard footnote markers BEFORE alias lookup: trailing */† are
# score-provenance footnotes (self-reported etc.), NOT model identity — kept
# in the name they split one model into marker-variant phantoms. Trailing-only.
named["model_version"] = (named["model_version"].astype(str)
                          .str.replace(r"\s*[*†]+\s*$", "", regex=True).str.strip())
named["benchmark"]     = named["benchmark"].map(lambda b: bench_map.get(b, b))
named["model_version"] = named["model_version"].map(lambda m: model_map.get(m, m))

# DROP sentinel: an empty canonical in model_aliases.csv means "remove from dataset".
# Used for uncensored or task-specific fine-tunes whose scores are not informative
# of a unified capability axis (Bio Llama, Hermes 3, Unsafety Llama).
drop_models = set(alias_df.loc[alias_df["canonical"].fillna("") == "", "variant"])
if drop_models:
    drop_mask = named["model_version_raw"].isin(drop_models)
    n_dropped = int(drop_mask.sum())
    n_models  = named.loc[drop_mask, "model_version_raw"].nunique()
    named = named[~drop_mask].reset_index(drop=True)
    print(f"DROP rule: removed {n_dropped} rows from {n_models} models flagged in alias map:")
    for m in sorted(drop_models):
        note = alias_df.loc[alias_df["variant"]==m, "note"].iloc[0]
        print(f"  - {m!r}: {note}")
else:
    print("No DROP-tagged models in alias map.")


name_changes = named[
    (named["benchmark"] != named["benchmark_raw"]) |
    (named["model_version"] != named["model_version_raw"])
][["model_version_raw", "model_version", "benchmark_raw", "benchmark", "_source_file"]].drop_duplicates()
name_changes.to_csv(OUTPUT_DIR / "name_changes.csv", index=False)
print(f"Wrote {len(name_changes)} rename records -> output/name_changes.csv")

# Models whose RESOLVED identity is neither aliased nor already reviewed. Keyed on
# the post-strip, post-alias name, not the raw one: a name differing only by a
# trailing */† footnote marker is the same test-taker (the strip above handles it),
# and a raw variant with an alias row is already a decided case. Keying on the raw
# name re-listed both every refresh, which buried the genuinely new models.
known_models = set(alias_df["variant"].unique()) | reviewed_models
new_models = sorted(set(named["model_version"].unique()) - known_models)
pd.DataFrame({"model_version": new_models}).to_csv(OUTPUT_DIR / "missing_models.csv", index=False)
print(f"{len(new_models)} models not in alias map -> output/missing_models.csv")

out_path = INTERMEDIATE_DIR / "03_scores_named.csv"
named.to_csv(out_path, index=False)
print(f"Wrote {out_path}")


DROP rule: removed 45 rows from 3 models flagged in alias map:
  - 'Llama 3.1 405b (Bio Llama)': DROP — RAND-specific bio fine-tune; confounds capability with task-specific post-training
  - 'Llama 3.1 405b (Hermes 3)': DROP — NousResearch chat fine-tune; capability axis ill-defined relative to base Instruct
  - 'Unsafety Llama': DROP — uncensored fine-tune; bio/chem scores reflect willingness to answer, not capability
Wrote 974 rename records -> output/name_changes.csv
22 models not in alias map -> output/missing_models.csv
Wrote /Users/yassineessifi/Desktop/ECI_Bayesian/1_data/1_pipeline/intermediate/03_scores_named.csv


## 05 — Dedup

Two passes:

1. **Exact bitwise duplicates** on `(model_version, benchmark, score)` —
   this is the load-bearing case (e.g. EnigmaEval 47×2).
   Always dropped.
2. **Disagreeing duplicates** on `(model_version, benchmark)` only — these
   shouldn't appear in practice, but if they do, resolve by `DEDUP_POLICY`
   (currently `"max"`). Every conflict goes to `output/duplicates_report.csv`
   with all original scores so the resolution is auditable.


In [10]:
before = len(named)

# Pass 1: exact duplicates
exact_keys = ["model_version", "benchmark", "score"]
n_exact = named.duplicated(subset=exact_keys, keep="first").sum()
deduped = named.drop_duplicates(subset=exact_keys, keep="first").copy()
print(f"Exact duplicates removed: {n_exact}  (before={before}, after={len(deduped)})")

# Pass 2: disagreeing duplicates on (model_version, benchmark)
disagree = (
    deduped.groupby(["model_version", "benchmark"])
    .agg(n=("score", "size"), score_min=("score", "min"), score_max=("score", "max"),
         sources=("_source_file", lambda s: ", ".join(sorted(set(s)))))
    .reset_index()
    .query("n > 1")
    .assign(spread=lambda d: d["score_max"] - d["score_min"])
    .sort_values("spread", ascending=False)
)

print(f"Disagreeing (model, benchmark) groups: {len(disagree)}")
if len(disagree):
    print(disagree.head(20).to_string(index=False))

# Section 03e's Kaggle rows carry a FIXED per-benchmark winner (KAGGLE_SPEC's
# kaggle_wins) rather than the generic max-score resolution below: on a
# (model, benchmark) collision the winner is chosen by source, not by which
# score happens to be larger. Every other benchmark defaults to priority 0 on
# both sides of a collision, so "max" still falls back to plain score there —
# this is a fixed-priority override for three benchmarks, not a new policy.
deduped["_kaggle_priority"] = 0
for _owner_slug, (_bench, _kaggle_wins) in KAGGLE_SPEC.items():
    _on_bench = deduped["benchmark"] == _bench
    _is_kaggle = _on_bench & (deduped["source"] == KAGGLE_SOURCE_FMT.format(_owner_slug))
    deduped.loc[_is_kaggle,            "_kaggle_priority"] = 1 if _kaggle_wins else 0
    deduped.loc[_on_bench & ~_is_kaggle, "_kaggle_priority"] = 0 if _kaggle_wins else 1

if DEDUP_POLICY == "max":
    deduped = (deduped.sort_values(["_kaggle_priority", "score"])
                      .drop_duplicates(subset=["model_version", "benchmark"], keep="last")
                      .drop(columns="_kaggle_priority"))
else:
    raise ValueError(f"Unknown DEDUP_POLICY: {DEDUP_POLICY}")

disagree.to_csv(OUTPUT_DIR / "duplicates_report.csv", index=False)
print(f"\nWrote output/duplicates_report.csv")
print(f"Final after dedup: {len(deduped):,} rows")

deduped.to_csv(INTERMEDIATE_DIR / "04_scores_deduped.csv", index=False)


Exact duplicates removed: 294  (before=5923, after=5629)
Disagreeing (model, benchmark) groups: 326
                    model_version            benchmark  n  score_min  score_max                    sources  spread
                   Llama-2-70b-hf                GSM8K  5     0.1330     0.6960         gsm8k_external.csv  0.5630
      gemini-3-deep-think-preview            ARC-AGI-2  2     0.4514     0.8458     arc_agi_2_external.csv  0.3944
         Meta-Llama-3-8B-Instruct           OpenBookQA  2     0.4500     0.8260  open_book_qa_external.csv  0.3760
                 Llama-2-13b-chat                GSM8K  2     0.0270     0.3690         gsm8k_external.csv  0.3420
                        PaLM 540B            ARC (AI2)  3     0.5300     0.8520       arc_ai2_external.csv  0.3220
              Baichuan-2-13B-Base                GSM8K  2     0.2210     0.5280         gsm8k_external.csv  0.3070
                         LLaMA-7B           OpenBookQA  3     0.2840     0.5720  open_book_qa_e

## 06 — Attach metadata (category)

Left-join `canonical/benchmark_metadata.csv` on the canonical benchmark
name. Seeded on initial setup from the existing
`1_data/processed/benchmarks_merged.csv` (56 benchmarks × constant per-benchmark
`category`). When new benchmarks appear, the join leaves
`category` NaN and they're listed below — add rows to the metadata file
by hand.

`category` labels MIRT factor loadings downstream (`ECIData.bench_category`). The
per-benchmark chance floor (`lower_bound`) is NOT carried here — its reviewed
ground truth lives in `1_data/curated/benchmark_lower_bounds.csv`, read directly
by the `--floors` 3PL fit; a second passthrough copy only drifts.


In [11]:
META_CSV = CANONICAL_DIR / "benchmark_metadata.csv"
meta = pd.read_csv(META_CSV)
print(f"Loaded {len(meta)} metadata rows from {META_CSV.name}")

merged = deduped.merge(
    meta[["benchmark", "category"]],
    on="benchmark", how="left",
)

missing_meta = sorted(merged[merged["category"].isna()]["benchmark"].unique())
if missing_meta:
    print(f"\n{len(missing_meta)} benchmark(s) missing metadata — please add rows to {META_CSV.name}:")
    for b in missing_meta:
        n = (merged["benchmark"] == b).sum()
        print(f"  - {b} ({n} rows)")
else:
    print("All benchmarks have metadata.")


Loaded 104 metadata rows from benchmark_metadata.csv
All benchmarks have metadata.


## 07 — Optional filter

`multiaxis_multiaxis_eci/data.py` already owns date / min-observation / category-drop filters; we
don't pre-filter on those. The one optional filter here is the curated
exclusion list (`1_data/curated/excluded_benchmarks.txt`), off by default to
match the canonical broad-index config.


In [12]:
EXCL_TXT = CURATED_DIR / "excluded_benchmarks.txt"

filtered = merged.copy()
if APPLY_CURATED_EXCLUSIONS and EXCL_TXT.exists():
    excl = [
        line.strip()
        for line in EXCL_TXT.read_text().splitlines()
        if line.strip() and not line.strip().startswith("#")
    ]
    before = len(filtered)
    filtered = filtered[~filtered["benchmark"].isin(excl)]
    print(f"Applied curated exclusions ({len(excl)} benchmarks): "
          f"before={before}, after={len(filtered)}")
else:
    print(f"APPLY_CURATED_EXCLUSIONS={APPLY_CURATED_EXCLUSIONS} — no filtering applied "
          f"(before={len(filtered)}, after={len(filtered)})")


APPLY_CURATED_EXCLUSIONS=False — no filtering applied (before=5052, after=5052)


## 08 — Humans

Read `1_data/curated/human_baselines.csv` (the project's authoritative file:
`[benchmark, group, score, note, source]`). Apply the section-04
canonicalization map to human-side benchmark spellings, keep only
benchmarks present in the fitted set, clip into `[ECI_EPS, 1-ECI_EPS]` so
`multiaxis_multiaxis_eci/data.py`'s open-interval filter doesn't drop saturated rows.


In [13]:
HUMANS_IN = CURATED_DIR / "human_baselines.csv"
humans = pd.read_csv(HUMANS_IN)
print(f"Loaded {len(humans)} human-baseline rows from {HUMANS_IN.name}")

humans["benchmark"] = humans["benchmark"].map(lambda b: bench_map.get(b, b))

fitted_benchmarks = set(filtered["benchmark"].unique())
before = len(humans)
humans = humans[humans["benchmark"].isin(fitted_benchmarks)].copy()
print(f"Filter to fitted benchmarks: before={before}, after={len(humans)}")

humans["score"] = humans["score"].clip(lower=ECI_EPS, upper=1 - ECI_EPS)

out_humans = OUTPUT_DIR / "human_baselines.csv"
humans.to_csv(out_humans, index=False)
print(f"Wrote {out_humans}")


Loaded 59 human-baseline rows from human_baselines.csv
Appended 13 RAND human-baseline rows; dropped 13 duplicate (benchmark, group) rows.
Filter to fitted benchmarks: before=59, after=59
Wrote /Users/yassineessifi/Desktop/ECI_Bayesian/1_data/1_pipeline/output/human_baselines.csv


## 09 — Assemble + content diff + report

Final `benchmarks_merged.csv` in the exact schema `multiaxis_multiaxis_eci/data.py` expects, then a
content diff vs the current `1_data/processed/benchmarks_merged.csv` (small
deltas on overlapping `(model, benchmark)` pairs = good), then a
`pipeline_report.md` summary.


In [14]:
# Match 1_data/processed/benchmarks_merged.csv schema exactly
SCHEMA_COLS = [
    "model_version", "score", "release_date", "organization",
    "benchmark", "stderr", "source", "category",
]

final = filtered.copy()
for col in SCHEMA_COLS:
    if col not in final.columns:
        if col == "stderr":
            final[col] = np.nan
        elif col == "source":
            final[col] = "EpochAI"
        elif col == "organization":
            final[col] = ""
        elif col == "release_date":
            final[col] = pd.NaT
        else:
            final[col] = np.nan

# Internal Epoch CSVs have no "Source" column; default missing source -> "EpochAI"
final["source"] = final["source"].fillna("EpochAI")
final = final[SCHEMA_COLS].copy()

# --- Canonicalize organization to one vendor per model (added 2026-07-06) ---
# Source feeds disagree on vendor casing (OpenAI/openai, DeepSeek/deepseek) and a
# few scraped SEAL rows are mis-attributed (MultiNRC tagged DeepSeek-R1 'openai'
# and a Claude row 'deepseek'). Map known case/synonym variants to a canonical
# vendor, then set each model's org to its MODAL canonical vendor: the lone
# mis-attributed outliers lose to the model's consensus, and casing collapses to
# one string per model. `organization` is not a fit input -- this is hygiene for
# per-vendor analysis and name-agnostic routing. Models with no org stay blank;
# multi-institution / academic strings are left untouched.
VENDOR_CANON = {
    "openai": "OpenAI", "anthropic": "Anthropic", "deepseek": "DeepSeek",
    "alibaba": "Alibaba", "mistral": "Mistral AI", "mistral ai": "Mistral AI",
    "meta": "Meta AI", "meta ai": "Meta AI", "moonshot": "Moonshot", "kimi": "Moonshot",
    "microsoft": "Microsoft", "amazon": "Amazon", "cohere": "Cohere",
    "minimax": "MiniMax", "xai": "xAI", "zai": "Z.ai (Zhipu AI)",
    "z.ai (zhipu ai)": "Z.ai (Zhipu AI)",
    "google": "Google DeepMind", "google deepmind": "Google DeepMind",
    "deepmind": "Google DeepMind", "google deepmind,google": "Google DeepMind",
    "google,google deepmind": "Google DeepMind",
}
def _canon_vendor(s):
    if not isinstance(s, str) or not s.strip():
        return s
    return VENDOR_CANON.get(s.strip().lower(), s)
final["organization"] = final["organization"].map(_canon_vendor)
def _modal_org(x):
    x = x.dropna()
    x = x[x.astype(str).str.strip() != ""]
    md = x.mode()
    return md.iloc[0] if len(md) else pd.NA
_modal = final.groupby("model_version")["organization"].agg(_modal_org)
final["organization"] = final["model_version"].map(_modal)

# Per-model override, for the cases the modal vote CANNOT rescue: outvoting a
# mis-attribution needs >=2 rows, so a model benchmarked on a single leaderboard
# keeps whatever that board said. Scale's Remote Labor Index labels every Manus
# release "DeepSeek"; Manus is built by Butterfly Effect (a.k.a. Monica).
MODEL_ORG_OVERRIDE = {
    "Manus 1.0":       "Butterfly Effect (Monica)",
    "Manus 1.5":       "Butterfly Effect (Monica)",
    "Manus_1.6 (Max)": "Butterfly Effect (Monica)",
}
_ovr = final["model_version"].map(MODEL_ORG_OVERRIDE)
final["organization"] = _ovr.fillna(final["organization"])
print(f"Org overrides: {int(_ovr.notna().sum())} rows over "
      f"{final.loc[_ovr.notna(), 'model_version'].nunique()} model(s)")
_n_org = int(final["organization"].notna().sum())
print(f"Org canonicalized: {final['organization'].dropna().nunique()} distinct vendors; "
      f"{_n_org} rows with org, {len(final) - _n_org} blank")

# --- Curated row-level drops (added 2026-07-05) ---
# 1_data/curated/row_drops_*.csv lists individual (model_version, benchmark)
# cells removed for data-correctness reasons (e.g. the bare "GPT-5.2" Epoch
# rows: an unlabeled config sitting below the model's own _low ladder — a
# config-mixture that can't belong to one test-taker). Applied here so a
# pipeline refresh doesn't silently resurrect them; each file carries a
# `reason` column as provenance.
drop_files = sorted(CURATED_DIR.glob("row_drops_*.csv"))
if drop_files:
    drop_keys = pd.concat([pd.read_csv(p)[["model_version", "benchmark"]]
                           for p in drop_files]).drop_duplicates()
    before_drops = len(final)
    final = final.merge(drop_keys.assign(_drop=True),
                        on=["model_version", "benchmark"], how="left")
    final = final[final["_drop"].isna()].drop(columns="_drop")
    print(f"Curated row drops: -{before_drops - len(final)} rows "
          f"({', '.join(p.name for p in drop_files)})")

# --- Curated row-level additions (added 2026-08-06) ---
# Mirror image of the drops glob above: 1_data/curated/row_adds_*.csv carries cells
# MEASURED LOCALLY that no feed supplies. Currently LAB-Bench Cloning, whose 31
# rows come from RAND RR-A3797-1 -- a published report that will never gain a
# model, so every 2025-2026 test-taker is permanently absent from that column and
# can only be filled by running the public set.
# Runs AFTER the drops so a cell that is both dropped and added stays dropped.
# Each row is full-schema plus a `reason` column, and its `source` names the
# harness and settings rather than the upstream feed: these rows are NOT the
# original protocol, and keeping them distinguishable is what lets sigma_b absorb
# the difference instead of hiding it.
add_files = sorted(CURATED_DIR.glob("row_adds_*.csv"))
if add_files:
    adds = pd.concat([pd.read_csv(f) for f in add_files], ignore_index=True)
    missing = [c for c in SCHEMA_COLS if c not in adds.columns]
    if missing:
        raise AssertionError(f"row_adds missing schema columns: {missing}")
    # A collision means a feed now supplies this cell, so the hand-added row is
    # stale. Raise rather than append: two rows for one (model, benchmark) would
    # otherwise be silently resolved by DEDUP_POLICY, picking by score.
    clash = adds.merge(final[["model_version", "benchmark"]].drop_duplicates(),
                       on=["model_version", "benchmark"])
    if len(clash):
        raise AssertionError(
            "row_adds collides with cells a feed now supplies -- retire the manual "
            f"row(s): {clash[['model_version', 'benchmark']].to_dict('records')}")
    final = pd.concat([final, adds[SCHEMA_COLS]], ignore_index=True)
    print(f"Curated row adds: +{len(adds)} rows "
          f"({', '.join(f.name for f in add_files)})")

out_merged = OUTPUT_DIR / "benchmarks_merged.csv"
final.to_csv(out_merged, index=False)
print(f"Wrote {len(final):,} rows -> {out_merged}")

# Content diff vs current file
current_path = PROCESSED_DIR / "benchmarks_merged.csv"
if current_path.exists():
    current = pd.read_csv(current_path)
    join_keys = ["model_version", "benchmark"]
    diff = (
        final[join_keys + ["score"]].rename(columns={"score": "score_new"})
        .merge(current[join_keys + ["score"]].rename(columns={"score": "score_old"}), on=join_keys)
        .assign(delta=lambda d: d["score_new"] - d["score_old"])
    )
    diff_sig = (diff[diff["delta"].abs() > 0.01]
                .assign(abs_delta=lambda d: d["delta"].abs())
                .sort_values("abs_delta", ascending=False)
                .drop(columns=["abs_delta"]))
    diff_sig.to_csv(OUTPUT_DIR / "content_diff_vs_current.csv", index=False)
    n_old = len(current.drop_duplicates(subset=join_keys))
    n_new = len(final.drop_duplicates(subset=join_keys))
    n_overlap = len(diff)
    print(f"\nCoverage vs {current_path.name}:")
    print(f"  current pairs: {n_old:,}")
    print(f"  new pairs:     {n_new:,}")
    print(f"  overlap:       {n_overlap:,}")
    print(f"  significant deltas (|delta|>0.01): {len(diff_sig):,}")
else:
    diff_sig = pd.DataFrame()
    n_old = n_new = n_overlap = 0
    print(f"\nNo current file at {current_path} to diff against.")


Org overrides: 3 rows over 3 model(s)
Org canonicalized: 62 distinct vendors; 4823 rows with org, 229 blank
Curated row drops: -5 rows (row_drops_2026-07-05.csv, row_drops_2026-07-28.csv, row_drops_2026-08-06.csv)
Curated row adds: +17 rows (row_adds_2026-08-06.csv, row_adds_2026-08-06_algotune.csv, row_adds_2026-08-06_arcprize.csv, row_adds_2026-08-07_aa.csv)
Wrote 5,064 rows -> /Users/yassineessifi/Desktop/ECI_Bayesian/1_data/1_pipeline/output/benchmarks_merged.csv

Coverage vs benchmarks_merged.csv:
  current pairs: 4,954
  new pairs:     5,064
  overlap:       4,894
  significant deltas (|delta|>0.01): 13


In [15]:
# Pipeline report
prov = json.loads(prov_path.read_text())
lines = []
lines.append(f"# Pipeline report — {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}")
lines.append("")
lines.append(f"- **Snapshot:** `{LATEST_SNAPSHOT.name}`")
lines.append(f"- **Epoch ZIP sha256:** `{prov['sha256']}`")
lines.append(f"- **Fetched at:** {prov['fetched_at']}")
lines.append("")
lines.append("## Row counts")
lines.append("")
lines.append(f"- After load:           {len(scores_long):,}")
lines.append(f"- After dedup:          {len(deduped):,}")
lines.append(f"- After metadata join:  {len(merged):,}")
lines.append(f"- After optional filter:{len(filtered):,}")
lines.append(f"- Final:                {len(final):,}")
lines.append("")
lines.append("## Distinct counts")
lines.append("")
lines.append(f"- Benchmarks: {final['benchmark'].nunique()}")
lines.append(f"- Models:     {final['model_version'].nunique()}")
lines.append("")
if n_old:
    lines.append("## Coverage vs current `1_data/processed/benchmarks_merged.csv`")
    lines.append("")
    lines.append(f"- Overlapping (model, benchmark) pairs: {n_overlap:,}")
    lines.append(f"- Current file pairs: {n_old:,}")
    lines.append(f"- New file pairs:     {n_new:,}")
    lines.append(f"- Significant score deltas (|Δ|>0.01): {len(diff_sig):,}")
    lines.append("")
    if len(diff_sig):
        lines.append("### Top 10 score changes")
        lines.append("")
        for _, r in diff_sig.head(10).iterrows():
            lines.append(f"- `{r['model_version']}` on `{r['benchmark']}`: "
                         f"{r['score_old']:.3f} → {r['score_new']:.3f} (Δ={r['delta']:+.3f})")
        lines.append("")
if live_unrecognized:
    lines.append(f"## Unrecognized live-feed tasks ({len(live_unrecognized)})")
    lines.append("")
    lines.append("In `epoch.ai/data/benchmarks.csv` but in neither `LIVE_TASK_SPEC` "
                 "nor `LIVE_TASK_SKIP` (section 03d). Map them or skip them with a reason:")
    for t in live_unrecognized:
        lines.append(f"- `{t}`")
    lines.append("")
if nan_dropped:
    tot_nan = sum(d["n"] for d in nan_dropped)
    lines.append(f"## Rows dropped for a blank `Model version` ({tot_nan})")
    lines.append("")
    lines.append("Valid scores with no versioned Epoch id, so no test-taker to key on. "
                 "A mix of real base models missing from Epoch's registry, agent "
                 "scaffolds, and task-specific fine-tunes. Recovering the first group "
                 "needs a curated `Name -> model_version` map.")
    lines.append("")
    for d in sorted(nan_dropped, key=lambda d: -d["n"]):
        ex = ", ".join(d["labels"][:3])
        lines.append(f"- `{d['file']}` ({d['benchmark']}): **{d['n']}** — e.g. {ex}")
    lines.append("")
if unrecognized:
    # Surfaced in the REPORT, not just cell output: these are new upstream files,
    # and the documented refresh workflow only says to read this file. Seven
    # benchmarks sat unmapped for at least one cycle because of that gap.
    lines.append(f"## Unrecognized upstream files ({len(unrecognized)})")
    lines.append("")
    lines.append("In the feed but in neither `FILE_SPEC` nor `FILE_SKIP`. "
                 "Map them or skip them with a reason:")
    for n in unrecognized:
        lines.append(f"- `{n}`")
    lines.append("")
if missing_meta:
    lines.append("## Benchmarks missing metadata")
    lines.append("")
    lines.append(f"Add rows to `canonical/benchmark_metadata.csv` for:")
    for b in missing_meta:
        lines.append(f"- {b}")
    lines.append("")
if new_models:
    lines.append(f"## New models since last canonicalization ({len(new_models)})")
    lines.append("")
    lines.append("See `output/missing_models.csv`. Add canonical aliases in "
                 "`canonical/model_aliases.csv` and re-run section 04.")
    lines.append("")

report_path = OUTPUT_DIR / "pipeline_report.md"
report_path.write_text("\n".join(lines))
print(f"Wrote {report_path}")


Wrote /Users/yassineessifi/Desktop/ECI_Bayesian/1_data/1_pipeline/output/pipeline_report.md


## 10 — Swap into the project tree

The outputs are the runtime files. `pipeline_report.md` stays the review
artifact; a bad refresh is undone with `git checkout 1_data/processed 1_data/curated`.


In [16]:
# Auto-swap: outputs become the runtime inputs (previously a manual cp step).
for src, dst in [
    (OUTPUT_DIR / "benchmarks_merged.csv", PROCESSED_DIR / "benchmarks_merged.csv"),
    (OUTPUT_DIR / "human_baselines.csv",   CURATED_DIR / "human_baselines.csv"),
]:
    shutil.copy(src, dst)
    print(f"swapped {src.name} -> {dst}")
print("\nReview output/pipeline_report.md; revert with: git checkout 1_data/processed 1_data/curated")


swapped benchmarks_merged.csv -> /Users/yassineessifi/Desktop/ECI_Bayesian/1_data/processed/benchmarks_merged.csv
swapped human_baselines.csv -> /Users/yassineessifi/Desktop/ECI_Bayesian/1_data/curated/human_baselines.csv

Review output/pipeline_report.md; revert with: git checkout 1_data/processed 1_data/curated
